In [1]:
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import csv
import time
import threading
from urllib.parse import quote, urlparse
from datetime import datetime
from calendar import monthrange
import requests
from bs4 import BeautifulSoup
import random
import pandas as pd
import os
from PIL import Image, ImageTk
from io import BytesIO
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException

class NaverMenuImageCollector:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("네이버 메뉴 이미지 수집기 - 스크롤다운 버전")
        self.root.geometry("1000x700")
        
        # 상태 변수
        self.running = False
        self.driver = None
        self.session = requests.Session()
        self.menus = []
        self.results = []
        self.total_images = 0
        
        self.setup_gui()
        self.setup_session()
        self.load_csv()
    
    def setup_session(self):
        """HTTP 세션 설정"""
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'ko-KR,ko;q=0.9,en;q=0.8',
            'Referer': 'https://www.naver.com/'
        })
        self.session.timeout = 15
    
    def setup_gui(self):
        """GUI 구성"""
        # 제목
        title_label = tk.Label(self.root, text="네이버 메뉴 이미지 수집기", 
                              font=("Arial", 16, "bold"))
        title_label.pack(pady=10)
        
        # CSV 상태
        self.csv_status = tk.Label(self.root, text="CSV 로딩 중...", font=("Arial", 10))
        self.csv_status.pack()
        
        # 기간 설정
        period_frame = tk.Frame(self.root)
        period_frame.pack(pady=10)
        
        tk.Label(period_frame, text="검색 월:").grid(row=0, column=0, padx=5)
        self.year_entry = tk.Entry(period_frame, width=6)
        self.year_entry.grid(row=0, column=1, padx=2)
        self.year_entry.insert(0, "2024")
        tk.Label(period_frame, text="년").grid(row=0, column=2)
        
        self.month_entry = tk.Entry(period_frame, width=4)
        self.month_entry.grid(row=0, column=3, padx=2)
        self.month_entry.insert(0, "6")
        tk.Label(period_frame, text="월").grid(row=0, column=4)
        
        # 버튼
        btn_frame = tk.Frame(self.root)
        btn_frame.pack(pady=10)
        
        self.start_btn = tk.Button(btn_frame, text="수집 시작", command=self.start_collection,
                                  bg="#27ae60", fg="white", width=10, height=2)
        self.start_btn.pack(side=tk.LEFT, padx=5)
        
        self.stop_btn = tk.Button(btn_frame, text="중지", command=self.stop_collection,
                                 bg="#e74c3c", fg="white", width=8, height=2, state=tk.DISABLED)
        self.stop_btn.pack(side=tk.LEFT, padx=5)
        
        # 현재 메뉴 및 통계
        stats_frame = tk.Frame(self.root)
        stats_frame.pack(pady=10)
        
        self.current_menu_label = tk.Label(stats_frame, text="현재: 대기중", 
                                          font=("Arial", 12, "bold"), fg="#e67e22")
        self.current_menu_label.pack()
        
        self.progress_bar = ttk.Progressbar(stats_frame, length=400)
        self.progress_bar.pack(pady=5)
        
        self.stats_label = tk.Label(stats_frame, text="메뉴: 0/0 | 매칭 이미지: 0개 | 전체 URL: 0개", 
                                   font=("Arial", 10))
        self.stats_label.pack()
        
        # 이미지 표시 영역
        image_frame = tk.LabelFrame(self.root, text="수집된 이미지 미리보기", font=("Arial", 11, "bold"))
        image_frame.pack(fill=tk.X, padx=10, pady=10)
        
        image_container = tk.Frame(image_frame)
        image_container.pack(pady=10)
        
        # 첫 번째 이미지
        first_frame = tk.Frame(image_container)
        first_frame.pack(side=tk.LEFT, padx=20)
        tk.Label(first_frame, text="첫 번째 이미지", font=("Arial", 10, "bold")).pack()
        self.first_image_label = tk.Label(first_frame, text="이미지 없음", width=25, height=12, 
                                         bg="lightgray", relief="solid")
        self.first_image_label.pack()
        
        # 통계 표시
        stats_center = tk.Frame(image_container)
        stats_center.pack(side=tk.LEFT, padx=20)
        tk.Label(stats_center, text="수집 결과", font=("Arial", 11, "bold")).pack()
        self.result_stats_label = tk.Label(stats_center, text="전체: 0개\n매칭: 0개\n일치율: 0%", 
                                          justify=tk.CENTER, font=("Arial", 10),
                                          bg="lightyellow", width=15, height=8, relief="solid")
        self.result_stats_label.pack()
        
        # 마지막 이미지
        last_frame = tk.Frame(image_container)
        last_frame.pack(side=tk.LEFT, padx=20)
        tk.Label(last_frame, text="마지막 이미지", font=("Arial", 10, "bold")).pack()
        self.last_image_label = tk.Label(last_frame, text="이미지 없음", width=25, height=12, 
                                        bg="lightgray", relief="solid")
        self.last_image_label.pack()
        
        # 로그
        log_frame = tk.LabelFrame(self.root, text="수집 로그", font=("Arial", 11, "bold"))
        log_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        self.log_text = scrolledtext.ScrolledText(log_frame, height=12, font=("Arial", 9))
        self.log_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
    
    def load_csv(self):
        """CSV 파일에서 상세메뉴 로드"""
        csv_file = '식당대12중53소132상세메뉴379분류.csv'
        
        try:
            if not os.path.exists(csv_file):
                self.csv_status.config(text="CSV 파일 없음", fg="red")
                return
            
            menus = []
            seen = set()
            
            with open(csv_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if '상세메뉴' in row and row['상세메뉴']:
                        detail_menus = [menu.strip() for menu in row['상세메뉴'].split(',')]
                        for menu in detail_menus:
                            if menu and menu not in seen and len(menu) >= 2:
                                seen.add(menu)
                                menus.append({
                                    'menu': menu,
                                    'category': f"{row.get('대분류', '')}/{row.get('중분류', '')}/{row.get('소분류', '')}"
                                })
            
            self.menus = menus
            self.csv_status.config(text=f"메뉴 {len(menus)}개 로드 완료", fg="green")
            self.log(f"CSV 로드 완료: {len(menus)}개 메뉴")
            
        except Exception as e:
            self.csv_status.config(text=f"CSV 로드 실패", fg="red")
            self.log(f"CSV 로드 오류: {e}")
    
    def log(self, message):
        """로그 출력"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.insert(tk.END, f"[{timestamp}] {message}\n")
        self.log_text.see(tk.END)
        self.root.update_idletasks()
    
    def setup_driver(self):
        """Selenium 드라이버 설정"""
        try:
            options = Options()
            options.add_argument('--headless')  # 백그라운드 실행
            options.add_argument('--no-sandbox')
            options.add_argument('--disable-dev-shm-usage')
            options.add_argument('--disable-gpu')
            options.add_argument('--window-size=1920,1080')
            
            self.driver = webdriver.Chrome(options=options)
            self.driver.implicitly_wait(10)
            self.log("Chrome 드라이버 준비 완료")
            return True
            
        except Exception as e:
            self.log(f"드라이버 설정 실패: {e}")
            return False
    
    def build_naver_url(self, menu_name, year, month):
        """네이버 이미지 검색 URL 구성"""
        last_day = monthrange(year, month)[1]
        start_date = f"{year}{month:02d}01"
        end_date = f"{year}{month:02d}{last_day:02d}"
        
        query = f"{menu_name} 음식"
        nso = f"so:r,p:from{start_date}to{end_date}"
        
        params = {
            'where': 'image',
            'query': query,
            'sm': 'tab_opt',
            'nso': nso,
            'mode': 'column',
            'section': 'image',
            'ccl': '0',
            'gif': '0'
        }
        
        param_parts = []
        for key, value in params.items():
            if key == 'nso':
                param_parts.append(f"{key}={quote(value, safe=':,')}")
            else:
                param_parts.append(f"{key}={quote(str(value))}")
        
        url = f"https://search.naver.com/search.naver?{'&'.join(param_parts)}"
        return url
    
    def scroll_and_collect_images(self, menu_name, year, month):
        """스크롤다운으로 모든 이미지 수집"""
        url = self.build_naver_url(menu_name, year, month)
        self.log(f"'{menu_name}' 검색 페이지 접속 중...")
        
        try:
            self.driver.get(url)
            time.sleep(3)
            
            self.log(f"'{menu_name}' 스크롤다운 시작...")
            
            # 스크롤다운으로 모든 이미지 로드
            last_height = 0
            stable_count = 0
            max_scrolls = 50  # 최대 스크롤 횟수
            scroll_count = 0
            
            while scroll_count < max_scrolls and stable_count < 5:
                # 현재 페이지 높이
                current_height = self.driver.execute_script("return document.body.scrollHeight")
                
                # 스크롤 다운
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)
                
                # 더보기 버튼 클릭 시도
                try:
                    more_button = self.driver.find_element(By.CSS_SELECTOR, ".btn_more, .more_wrap .btn")
                    if more_button.is_displayed():
                        more_button.click()
                        time.sleep(2)
                except:
                    pass
                
                # 높이 변화 확인
                new_height = self.driver.execute_script("return document.body.scrollHeight")
                if new_height == current_height:
                    stable_count += 1
                else:
                    stable_count = 0
                
                scroll_count += 1
                
                if scroll_count % 5 == 0:
                    self.log(f"'{menu_name}' 스크롤 진행 중... ({scroll_count}회)")
            
            self.log(f"'{menu_name}' 스크롤 완료, 이미지 수집 중...")
            
            # 모든 이미지 요소 수집
            image_elements = self.driver.find_elements(By.CSS_SELECTOR, "img[src*='http']")
            
            all_images = []
            seen_urls = set()
            
            for img_element in image_elements:
                try:
                    src = img_element.get_attribute('src')
                    alt = img_element.get_attribute('alt') or ''
                    title = img_element.get_attribute('title') or ''
                    
                    if src and src not in seen_urls and self.is_valid_image_url(src):
                        seen_urls.add(src)
                        all_images.append({
                            'url': src,
                            'alt': alt,
                            'title': title,
                            'menu': menu_name
                        })
                        
                except Exception:
                    continue
            
            self.log(f"'{menu_name}' 전체 {len(all_images)}개 이미지 발견")
            
            # 메뉴명이 포함된 이미지만 필터링
            matched_images = self.filter_images_by_menu_name(all_images, menu_name)
            
            self.log(f"'{menu_name}' 매칭 이미지: {len(matched_images)}개")
            
            return all_images, matched_images
            
        except Exception as e:
            self.log(f"'{menu_name}' 수집 오류: {e}")
            return [], []
    
    def is_valid_image_url(self, url):
        """유효한 이미지 URL 확인"""
        if not url or len(url) < 20:
            return False
            
        url_lower = url.lower()
        
        # 네이버 도메인 확인
        valid_domains = ['pstatic.net', 'blogfiles.naver.net', 'postfiles.naver.net', 'cafephinf.pstatic.net']
        if not any(domain in url_lower for domain in valid_domains):
            return False
        
        # 제외 키워드
        exclude_keywords = ['icon', 'logo', 'ad', 'banner', 'button', 'nav']
        if any(keyword in url_lower for keyword in exclude_keywords):
            return False
        
        return True
    
    def filter_images_by_menu_name(self, images, menu_name):
        """이미지 제목에 상세메뉴가 포함된 것만 필터링"""
        matched_images = []
        menu_lower = menu_name.lower()
        
        for img in images:
            alt_text = img.get('alt', '').lower()
            title_text = img.get('title', '').lower()
            combined_text = f"{alt_text} {title_text}"
            
            # 메뉴명이 포함되어 있는지 확인
            if menu_lower in combined_text:
                img['match_type'] = '정확매칭'
                matched_images.append(img)
            else:
                # 부분 매칭 (2글자 이상)
                if len(menu_name) >= 2:
                    for i in range(len(menu_name) - 1):
                        substring = menu_name[i:i+2].lower()
                        if substring in combined_text:
                            img['match_type'] = '부분매칭'
                            matched_images.append(img)
                            break
        
        return matched_images
    
    def load_and_display_image(self, url, label, size=(200, 150)):
        """이미지 로드 및 표시"""
        try:
            response = self.session.get(url, timeout=10)
            response.raise_for_status()
            
            image = Image.open(BytesIO(response.content))
            image = image.convert('RGB')
            image.thumbnail(size, Image.Resampling.LANCZOS)
            
            photo = ImageTk.PhotoImage(image)
            label.config(image=photo, text="")
            label.image = photo
            
        except Exception as e:
            label.config(text=f"로드 실패\n{str(e)[:20]}...", bg="lightcoral")
    
    def update_image_display(self, matched_images):
        """첫 번째/마지막 이미지 표시 업데이트"""
        # 이미지 레이블 초기화
        self.first_image_label.config(image="", text="이미지 없음", bg="lightgray")
        self.last_image_label.config(image="", text="이미지 없음", bg="lightgray")
        
        if matched_images:
            # 첫 번째 이미지
            first_img = matched_images[0]
            self.first_image_label.config(text="로딩 중...", bg="lightyellow")
            threading.Thread(target=self.load_and_display_image, 
                           args=(first_img['url'], self.first_image_label), daemon=True).start()
            
            # 마지막 이미지 (첫 번째와 다른 경우)
            if len(matched_images) > 1:
                last_img = matched_images[-1]
                self.last_image_label.config(text="로딩 중...", bg="lightyellow")
                threading.Thread(target=self.load_and_display_image, 
                               args=(last_img['url'], self.last_image_label), daemon=True).start()
            else:
                self.last_image_label.config(text="첫 번째와 동일", bg="lightblue")
    
    def start_collection(self):
        """수집 시작"""
        if not self.menus:
            messagebox.showerror("오류", "로드된 메뉴가 없습니다.")
            return
        
        try:
            year = int(self.year_entry.get())
            month = int(self.month_entry.get())
            
            if not (1 <= month <= 12):
                messagebox.showerror("오류", "월은 1-12 사이여야 합니다.")
                return
                
        except ValueError:
            messagebox.showerror("오류", "년도와 월을 정확히 입력하세요.")
            return
        
        # 상태 변경
        self.running = True
        self.start_btn.config(state=tk.DISABLED)
        self.stop_btn.config(state=tk.NORMAL)
        self.results = []
        self.total_images = 0
        
        # 로그 초기화
        self.log_text.delete(1.0, tk.END)
        self.log("=" * 60)
        self.log("네이버 메뉴 이미지 수집 시작 (스크롤다운 방식)")
        self.log(f"검색 기간: {year}년 {month}월")
        self.log(f"대상 메뉴: {len(self.menus)}개")
        self.log("=" * 60)
        
        # 진행률 설정
        self.progress_bar['maximum'] = len(self.menus)
        self.progress_bar['value'] = 0
        
        # 수집 스레드 시작
        thread = threading.Thread(target=self.run_collection, args=(year, month))
        thread.daemon = True
        thread.start()
    
    def run_collection(self, year, month):
        """실제 수집 실행"""
        # 드라이버 설정
        if not self.setup_driver():
            self.log("드라이버 설정 실패로 수집을 중단합니다.")
            self.cleanup()
            return
        
        try:
            for i, menu_data in enumerate(self.menus, 1):
                if not self.running:
                    break
                
                menu_name = menu_data['menu']
                
                # 상태 업데이트
                self.current_menu_label.config(text=f"수집 중: {menu_name}")
                self.progress_bar['value'] = i
                
                # 이미지 수집
                all_images, matched_images = self.scroll_and_collect_images(menu_name, year, month)
                
                if matched_images:
                    # 결과 저장
                    self.results.append({
                        'menu': menu_name,
                        'category': menu_data['category'],
                        'year': year,
                        'month': month,
                        'total_images': len(all_images),
                        'matched_images': len(matched_images),
                        'match_rate': len(matched_images) / len(all_images) * 100 if all_images else 0,
                        'image_urls': [img['url'] for img in matched_images],
                        'image_details': matched_images
                    })
                    
                    self.total_images += len(matched_images)
                    
                    # 이미지 표시 업데이트
                    self.update_image_display(matched_images)
                    
                    # 통계 업데이트
                    match_rate = len(matched_images) / len(all_images) * 100 if all_images else 0
                    stats_text = f"전체: {len(all_images)}개\n매칭: {len(matched_images)}개\n일치율: {match_rate:.1f}%"
                    self.result_stats_label.config(text=stats_text)
                
                # 전체 통계 업데이트
                self.stats_label.config(text=f"메뉴: {i}/{len(self.menus)} | 매칭 이미지: {self.total_images}개 | 전체 URL: {sum(len(r['image_urls']) for r in self.results)}개")
                
                self.log(f"{menu_name}: 전체 {len(all_images)}개 -> 매칭 {len(matched_images)}개 URL 수집")
                
                # 메뉴 간 딜레이
                if self.running and i < len(self.menus):
                    time.sleep(2)
            
            if self.running:
                self.current_menu_label.config(text="수집 완료")
                self.log("=" * 60)
                self.log("수집 완료!")
                self.log(f"총 {len(self.results)}개 메뉴 처리")
                self.log(f"총 {self.total_images}개 매칭 이미지 URL 수집")
                self.log("=" * 60)
                self.save_results_to_excel()
            else:
                self.current_menu_label.config(text="수집 중단됨")
                self.log("수집이 중단되었습니다.")
                
        except Exception as e:
            self.log(f"수집 중 오류: {e}")
        finally:
            self.cleanup()
    
    def save_results_to_excel(self):
        """결과를 엑셀로 저장"""
        if not self.results:
            return
        
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            year = self.results[0]['year']
            month = self.results[0]['month']
            filename = f"네이버_메뉴이미지_수집결과_{year}{month:02d}_{timestamp}.xlsx"
            
            # 요약 데이터
            summary_data = []
            url_data = []
            
            for result in self.results:
                summary_data.append({
                    '메뉴명': result['menu'],
                    '분류': result['category'],
                    '년도': result['year'],
                    '월': result['month'],
                    '전체이미지': result['total_images'],
                    '매칭이미지': result['matched_images'],
                    '일치율(%)': f"{result['match_rate']:.1f}%",
                    'URL개수': len(result['image_urls'])
                })
                
                # URL 상세 데이터
                for i, img_detail in enumerate(result['image_details'], 1):
                    url_data.append({
                        '메뉴명': result['menu'],
                        '순번': i,
                        '이미지URL': img_detail['url'],
                        'ALT텍스트': img_detail.get('alt', ''),
                        'TITLE텍스트': img_detail.get('title', ''),
                        '매칭타입': img_detail.get('match_type', ''),
                        '도메인': urlparse(img_detail['url']).netloc
                    })
            
            # 엑셀 저장
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                pd.DataFrame(summary_data).to_excel(writer, sheet_name='수집요약', index=False)
                if url_data:
                    pd.DataFrame(url_data).to_excel(writer, sheet_name='URL상세', index=False)
            
            self.log(f"결과 저장 완료: {filename}")
            
        except Exception as e:
            self.log(f"결과 저장 실패: {e}")
    
    def stop_collection(self):
        """수집 중지"""
        self.running = False
        self.log("수집 중지 요청")
    
    def cleanup(self):
        """정리"""
        self.running = False
        
        if self.driver:
            try:
                self.driver.quit()
                self.log("브라우저 종료")
            except:
                pass
            self.driver = None
        
        self.start_btn.config(state=tk.NORMAL)
        self.stop_btn.config(state=tk.DISABLED)
    
    def run(self):
        """앱 실행"""
        def on_closing():
            if self.running:
                if messagebox.askokcancel("종료", "수집 중입니다. 종료하시겠습니까?"):
                    self.stop_collection()
                    time.sleep(1)
                    self.root.destroy()
            else:
                self.root.destroy()
        
        self.root.protocol("WM_DELETE_WINDOW", on_closing)
        self.root.mainloop()

def main():
    app = NaverMenuImageCollector()
    app.run()

if __name__ == "__main__":
    main()